# ClearWater-Modules Demo 2: Coupling Water Quality Reactions to Transport with ClearWater-Riverine

**Objective**: Demonstrate a more complex scenario of coupled transport and reaction models in Sumwere Creek, using the [ClearWater-modules](https://github.com/EcohydrologyTeam/ClearWater-modules) to simulate heat exchange with the atmosphere.

This notebook builds on the introduction to using [ClearWater-riverine](https://github.com/EcohydrologyTeam/ClearWater-riverine) provided in the demo notebook in that reposittory.

## Background 
This notebook couples Clearwater-riverine (transport) with Clearwater-modules (reactions) - specifically, the Temperature Simulation Model (TSM). The Temperature Simulation Module (TSM) is an essential component of ClearWater (Corps Library for Environmental Analysis and Restoration of Watersheds). TSM plays a crucial role in simulating and predicting water temperature within aquatic ecosystems. TSM utilizes a comprehensive energy balance approach to account for various factors contributing to heat inputs and outputs in the water environment. It considers both external forcing functions and heat exchanges occurring at the water surface and the sediment-water interface. The primary contributors to heat exchange at the water surface include shortwave solar radiation, longwave atmospheric radiation, heat conduction from the atmosphere to the water, and direct heat inputs. Conversely, the primary factors that remove heat from the system are longwave radiation emitted by the water, evaporation, and heat conduction from the water to the atmosphere. 
The core principle behind TSM is the application of the laws of conservation of energy to compute water temperature. This means that the change in heat content of the water is directly related to changes in temperature, which, in turn, are influenced by various heat flux components. The specific heat of water is employed to establish this relationship. Each term of the heat flux equation can be calculated based on the input provided by the user, allowing for flexibility in modeling different environmental conditions

## Example Case Study

This example shows how to run Clearwater Riverine coupled with Clearwater Modules in a fictional location, "Sumwere Creek" (shown below). The flow field for Sumwere Creek comes from a HEC-RAS 2D model, which has a domain of 2x2 km and a base mesh cell size of 100x100 meters. 

![image.png](../docs/imgs/SumwereCreek_coarse.png)

The upstream boundary for Sumwere Creek is at the top left of the model domain, flowing into the domain at a constant 3 cms. At the first bend in the creek, there is an additional boundary representing a spring-fed tributary to the creek (1 cms). Further downstream, there is a meander in the stream forming a slow-flowing oxbow lake. There is another boundary flowing into that oxbow lake, representing a powerplant discharge (0.5 cms). 

The downstream boundary is a constant stage set at 20.75. The upstream inflows have a water temperature of 15 degrees C; the spring-fed creek has constant inflows of 10 C, and the powerplant is steady at 20 C with periodic higher temperature (25 C) discharges in a downstream meander.  

We simulate this scenario over the course of two full days, using meteorological parameters from Arizona (extreme temperature swings between night and day) to help show off the impacts of TSM.

## Data Availability
All data required run this notebook is available at this [Google Drive](https://drive.google.com/drive/folders/19uCjAJPZh4g6r1ZWzk1D_B8jZGluSc4N?usp=drive_link). 
This notebook will use the version two `sumwere_creek_coarse_p48` model. Please download that entire folder and place it in the `data_temp` folder (`examples/data_temp`) of this repository to run the rest of the notebook. 

Alternatively, if you would like to run a different version of the model (see the [ReadMe](https://docs.google.com/document/d/1FKjrTZHUYmYxo0mgn72dOezHtq-CFR86rQ1ObD1ZY0c/edit) for details), download that folder, place it in the `data_temp` folder. You may need to adjust path names in the notebook accordingly.

## Model Set-Up
### General Imports

add something here about the pixi environment setup...

In [1]:
from pathlib import Path
from typing import Optional

import pandas as pd
import xarray as xr

import hvplot.xarray
import hvplot.pandas
import holoviews as hv
from bokeh.models import HoverTool

from clearwater_modules_v2.config import init_from_file
from clearwater_riverine.plotting import RiverinePlotter

### Instantiate Models
#### Clearwater-Modules & Clearwater-Riverine

Ensure that you have followed the instructions in the Data Availability Section, and that you have all files downloaded from the [Google Drive](https://drive.google.com/drive/folders/19uCjAJPZh4g6r1ZWzk1D_B8jZGluSc4N?usp=drive_link) for version 2 `sumwere_creek_coarse_p48` and saved/unzipped to your local directory `examples/data_temp`. For a more detailed explanation of all the steps in this process, please see [01_getting_started_riverine.ipynb](./01_getting_started_riverine.ipynb).

This example sets up the model using a config file. The config files are structured to simulate "water_temperature" and "water_temperature_mixed". The output variable "water_temperature" are the model results from the linked Riverine/TSM simulation and represents the transport/mixing from the Riverine model in addition to the comprehensive energy balance from the TSM model. While, the output variable "water_temperature_mixed" represents only the transport/mixing from the Riverine model.

In [2]:
# Find project directory (i.e. the parent to `/examples` directory for this notebook)
project_path = Path.cwd().parent
project_path

WindowsPath('D:/Clearwater/ClearWater-modules')

In [3]:
#### Set local filepath to the configuration yaml file ####
model_name = 'sumwere_creek_coarse_p48'
test_case_path = project_path / 'examples/data_temp' / model_name

config_filename = 'modules.yml'
config_path = test_case_path / config_filename
#config_path = Path(r"D:\Clearwater\ClearWater-modules\examples\data_temp\sumwere_creek_coarse_p48\modules.yml")

print(config_path.exists())

True


In [4]:
#### Initialize a version 2 model of the linked Riverine and TSM models ####
model = init_from_file(config_path)

### Run the Coupled Models

In [5]:
#### Simulate a version 2 model of the linked Riverine and TSM models ####
model.run()

[2026-04-08 17:11:21] INFO - Running timestep: 2022-05-13 08:00:00
[2026-04-08 17:11:21] INFO - Running timestep: 2022-05-13 08:00:30
[2026-04-08 17:11:21] INFO - Running timestep: 2022-05-13 08:01:00
[2026-04-08 17:11:22] INFO - Running timestep: 2022-05-13 08:01:30
[2026-04-08 17:11:22] INFO - Running timestep: 2022-05-13 08:02:00
[2026-04-08 17:11:22] INFO - Running timestep: 2022-05-13 08:02:30
[2026-04-08 17:11:22] INFO - Running timestep: 2022-05-13 08:03:00
[2026-04-08 17:11:22] INFO - Running timestep: 2022-05-13 08:03:30
[2026-04-08 17:11:22] INFO - Running timestep: 2022-05-13 08:04:00
[2026-04-08 17:11:22] INFO - Running timestep: 2022-05-13 08:04:30
[2026-04-08 17:11:22] INFO - Running timestep: 2022-05-13 08:05:00
[2026-04-08 17:11:22] INFO - Running timestep: 2022-05-13 08:05:30
[2026-04-08 17:11:22] INFO - Running timestep: 2022-05-13 08:06:00
[2026-04-08 17:11:22] INFO - Running timestep: 2022-05-13 08:06:30
[2026-04-08 17:11:22] INFO - Running timestep: 2022-05-13 08:0

### Plot the Coupled Models Results

In [6]:
#Initialize a Riverine dynamic plotting tool
plotter = RiverinePlotter(registry=model._Model__registry, crs='EPSG:26916')

In [7]:
#Plot the water temperature results from the linked Riverine and TSM model simulation
plotter.dynamic_plot(constituent_name = 'water_temperature')

:DynamicMap   [datetime]
   :Overlay
      .Polygons.I :Polygons   [Longitude,Latitude]   (water_temperature,nface)
      .WMTS.I     :WMTS   [Longitude,Latitude]

In [8]:
#Plot the water temperature results from just the Riverine model simulation
plotter.dynamic_plot(constituent_name = 'water_temperature_mixed')

:DynamicMap   [datetime]
   :Overlay
      .Polygons.I :Polygons   [Longitude,Latitude]   (water_temperature_mixed,nface)
      .WMTS.I     :WMTS   [Longitude,Latitude]

In [9]:
# defaults for line plot
cells = {
    217: 'upstream reach, midway', 
    285: 'upstream reach just before spring',
    226: 'confluence of creek and spring',
    151: 'mixing zone power plant',  
    180: 'further from power plant',
    317: 'confluence below power plant',
}

In [10]:
def model_compare_vars_lines_plots(
    model: clearwater_modules_v2.model.Model, # model instance
    variable_name_1: str, 
    variable_name_2: str,
    cells_dict: dict[int, str], #with key as cell intergers and value as cell location used for plot titles
):
    '''Holoviews line plot overlay for a given variable at selected grid cells.'''
    
    layout_curve_plots_list = []
    
    for cell, title_name in cells_dict.items():
        overlay_curve_plots_list = []
        ds_var1 = model._Model__registry.get(variable_name_1).isel(nface=cell)
        ds_var2 = model._Model__registry.get(variable_name_2).isel(nface=cell)
        curve_plot_1 = hv.Curve(ds_var1, label=f'cell {cell}; var1').opts(tools=['hover'])
        curve_plot_2 = hv.Curve(ds_var2, label=f'cell {cell}; var2').opts(tools=['hover'])
        overlay_curve_plots_list.append(curve_plot_1)
        overlay_curve_plots_list.append(curve_plot_2)        
        overlay_plot_i = hv.Overlay(overlay_curve_plots_list).opts(width=400, height=300, legend_position='top_left', title=title_name)
        layout_curve_plots_list.append(overlay_plot_i)

    return hv.Layout(layout_curve_plots_list).cols(1)

In [11]:
model_compare_vars_lines_plots(model=model, variable_name_1 = 'water_temperature', variable_name_2 = 'water_temperature_mixed', cells_dict=cells)

:Layout
   .Overlay.I   :Overlay
      .Curve.Cell_217_semicolon_var1 :Curve   [time]   (water_temperature)
      .Curve.Cell_217_semicolon_var2 :Curve   [time]   (water_temperature_mixed)
   .Overlay.II  :Overlay
      .Curve.Cell_285_semicolon_var1 :Curve   [time]   (water_temperature)
      .Curve.Cell_285_semicolon_var2 :Curve   [time]   (water_temperature_mixed)
   .Overlay.III :Overlay
      .Curve.Cell_226_semicolon_var1 :Curve   [time]   (water_temperature)
      .Curve.Cell_226_semicolon_var2 :Curve   [time]   (water_temperature_mixed)
   .Overlay.IV  :Overlay
      .Curve.Cell_151_semicolon_var1 :Curve   [time]   (water_temperature)
      .Curve.Cell_151_semicolon_var2 :Curve   [time]   (water_temperature_mixed)
   .Overlay.V   :Overlay
      .Curve.Cell_180_semicolon_var1 :Curve   [time]   (water_temperature)
      .Curve.Cell_180_semicolon_var2 :Curve   [time]   (water_temperature_mixed)
   .Overlay.VI  :Overlay
      .Curve.Cell_317_semicolon_var1 :Curve   [time]   (water_temperature)
      .Curve.Cell_317_semicolon_var2 :Curve   [time]   (water_temperature_mixed)